# Credit Card Complaint Analysis: Python + Power BI

**Independent portfolio project | CFPB public complaints received in 2024**

**Workflow:** Using Jupyter and pandas to inspect, clean, validate, and analyze the data. Export one clean CSV & Use Power BI for the visuals.

## 1. Import and inspecting the data

Checking the number of rows, the available fields, and a small sample. The public CFPB dataset records published complaints; it does not represent all cardholders.

In [1]:
from pathlib import Path
import pandas as pd

source = Path("data/credit_card_complaints_2024.csv")
if not source.exists():
    source = Path("../data/credit_card_complaints_2024.csv")
df = pd.read_csv(source, low_memory=False)
print("Rows and columns:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

Rows and columns: (75989, 15)


,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2024-01-01T01:53:02.000Z,Credit card,General-purpose credit card or charge card,Improper use of your report,Reporting company used your report improperly,NaN,CAPITAL ONE FINANCIAL CORPORATION,GA,31537,NaN,Web,2024-01-01T02:02:02.000Z,Closed with explanation,Yes,8085114
1,2024-01-01T06:23:49.000Z,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94538,NaN,Web,2024-01-01T07:09:13.000Z,Closed with explanation,Yes,8086913
2,2024-01-01T13:57:28.000Z,Credit card,Store credit card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,Company has responded to the consumer and the ...,SYNCHRONY FINANCIAL,NJ,076XX,NaN,Web,2024-01-10T15:35:28.000Z,Closed with explanation,Yes,8086654
3,2024-01-01T15:48:58.000Z,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Problem with personal statement of dispute,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",IN,46235,NaN,Web,2024-01-01T16:12:17.000Z,Closed with explanation,Yes,8086469
4,2024-01-01T16:29:32.000Z,Credit card,Store credit card,Fees or interest,Problem with fees,NaN,TD BANK US HOLDING COMPANY,GA,30075,NaN,Web,2024-01-01T16:39:47.000Z,Closed with explanation,Yes,8086422


Columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']


## 2. Choose fields and check quality

Keeping only fields relevant to the three questions. Avoiding complaint narratives and ZIP codes. Checking missing values and repeated complaint IDs before cleaning.

In [2]:
df.isna().sum()

Date received                       0
Product                             0
Sub-product                         0
Issue                               0
Sub-issue                          63
Company public response         37517
Company                             0
State                             213
ZIP code                            0
Tags                            65162
Submitted via                       0
Date sent to company                0
Company response to consumer        0
Timely response?                    0
Complaint ID                        0
dtype: int64

In [4]:
columns = ["Complaint ID", "Date received", "Product", "Issue", "Sub-issue",
           "Company", "State", "Company response to consumer", "Timely response?"]
work = df[columns].copy()
print("Missing values before cleaning:")
display(work.isna().sum().to_frame("missing_rows"))
print("Repeated complaint IDs:", work["Complaint ID"].duplicated().sum())
print("Products:")
display(work["Product"].value_counts(dropna=False))

Missing values before cleaning:


,missing_rows
Complaint ID,0
Date received,0
Product,0
Issue,0
Sub-issue,63
Company,0
State,213
Company response to consumer,0
Timely response?,0


Repeated complaint IDs: 0
Products:


Product
Credit card    75989
Name: count, dtype: int64

In [3]:
df["Complaint ID"].duplicated().sum()

np.int64(0)

In [5]:
work["State"] = work["State"].fillna("Unknown")
work["State"].isna().sum()

np.int64(0)

## 3. Cleaning the data

Parse dates,and removing records without an ID/date/issue,duplicate IDs, and labeling missing state or response as `Unknown`. Keeping missing state records for national totals.

In [6]:
work["Sub-issue"] = work["Sub-issue"].fillna("Unknown")
work["Sub-issue"].isna().sum()

np.int64(0)

In [7]:
work = work.rename(columns={
    "Complaint ID": "complaint_id", "Date received": "date_received",
    "Product": "product", "Issue": "issue", "Sub-issue": "sub_issue",
    "Company": "company", "State": "state",
    "Company response to consumer": "response", "Timely response?": "timely_response"
})
work["date_received"] = pd.to_datetime(work["date_received"], errors="coerce", utc=True)
for column in ["product", "issue", "sub_issue", "company", "state", "response", "timely_response"]:
    work[column] = work[column].astype("string").str.strip()
work = work.loc[work["product"].eq("Credit card") & work["date_received"].dt.year.eq(2024)]
work = work.dropna(subset=["complaint_id", "date_received", "issue"])
work = work.drop_duplicates(subset="complaint_id").copy()
work["state"] = work["state"].fillna("Unknown")
work["response"] = work["response"].fillna("Unknown")
work["month"] = work["date_received"].dt.strftime("%Y-%m")
print("Rows after cleaning:", len(work))
print("Unique IDs:", work["complaint_id"].nunique())
print("Date range:", work["date_received"].min().date(), "to", work["date_received"].max().date())
assert len(work) == work["complaint_id"].nunique()
assert work["date_received"].dt.year.eq(2024).all()

Rows after cleaning: 75989
Unique IDs: 75989
Date range: 2024-01-01 to 2024-12-31


## 4. Question 1: Which issues appear most often?

Using pandas to calculate counts and shares. 

In [8]:
issues = work["issue"].value_counts()
issue_table = issues.head(10).rename_axis("issue").reset_index(name="complaints")
issue_table["share_pct"] = (issue_table["complaints"] / len(work) * 100).round(2)
display(issue_table)

,issue,complaints,share_pct
0,Problem with a purchase shown on your statement,14595,19.21
1,Incorrect information on your report,13261,17.45
2,Problem with a company's investigation into an...,9911,13.04
3,Getting a credit card,9683,12.74
4,"Other features, terms, or problems",6769,8.91
5,Fees or interest,6343,8.35
6,Closing your account,4273,5.62
7,Problem when making payments,4147,5.46
8,"Advertising and marketing, including promotion...",2766,3.64
9,Trouble using your card,2315,3.05


## 5. Question 2: How did monthly complaint volume change?

These are month-by-month counts, not rates adjusted for cardholders or days in each month.

In [9]:
monthly = work.groupby("month").size().rename("complaints").reset_index()
display(monthly)
print("Highest-volume month:", monthly.loc[monthly["complaints"].idxmax()].to_dict())

,month,complaints
0,2024-01,5247
1,2024-02,5182
2,2024-03,5889
3,2024-04,6676
4,2024-05,6760
5,2024-06,5535
6,2024-07,6192
7,2024-08,7355
8,2024-09,7185
9,2024-10,7176


Highest-volume month: {'month': '2024-08', 'complaints': 7355}


## 6. Question 3: How do response types vary across common issues?

Comparing response **shares within each issue**. These administrative categories do not measure consumer satisfaction or whether a complaint was valid.

In [10]:
top_issues = issues.head(5).index
selected = work.loc[work["issue"].isin(top_issues)]
response_counts = pd.crosstab(selected["issue"], selected["response"])
response_share = response_counts.div(response_counts.sum(axis=1), axis=0).mul(100).round(1)
display(response_counts)
display(response_share)

response,Closed with explanation,Closed with monetary relief,Closed with non-monetary relief,Untimely response
issue,,,,
Getting a credit card,6290,155,3233,5
Incorrect information on your report,6647,78,6534,2
"Other features, terms, or problems",4743,1043,979,4
Problem with a company's investigation into an existing problem,4780,59,5072,0
Problem with a purchase shown on your statement,8794,4612,1185,4


response,Closed with explanation,Closed with monetary relief,Closed with non-monetary relief,Untimely response
issue,,,,
Getting a credit card,65.0,1.6,33.4,0.1
Incorrect information on your report,50.1,0.6,49.3,0.0
"Other features, terms, or problems",70.1,15.4,14.5,0.1
Problem with a company's investigation into an existing problem,48.2,0.6,51.2,0.0
Problem with a purchase shown on your statement,60.3,31.6,8.1,0.0


## 7. Export for Power BI

Save the cleaned records as CSV. Power BI will use this file to create the visuals. The `month` column supports a monthly trend chart.

In [11]:
output = source.parent / "credit_card_complaints_2024_clean.csv"
work["date_received"] = work["date_received"].dt.strftime("%Y-%m-%d")
work.to_csv(output, index=False)
print("Saved:", output.resolve())
print("Rows exported:", len(work))

Saved: C:\Users\Owner\Downloads\CFPB_Credit_Card_Complaints_Project_Starter (3)\cfpb-python-notebook\data\credit_card_complaints_2024_clean.csv
Rows exported: 75989


## 8. Interpretation prompts

After viewing the Power BI dashboard, write three observations for the README:

1. Which issue has the highest count and what share does it represent?
2. Which month has the highest complaint count?
3. What response category is most common within the leading issue?

Published complaints are not a measure of all cardholders, verified violations, or consumer satisfaction. Company counts should not be interpreted as rates without a customer-base denominator.